In [1]:
import io
import chess.pgn
import pandas as pd
import numpy as np
import requests
import zstandard 
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone, BaseEstimator, TransformerMixin
from stockfish import Stockfish
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from concurrent.futures import ThreadPoolExecutor
import os


In [2]:
SF_PATH = r"C:\Users\yazid\Downloads\stockfish-windows-x86-64-universal\stockfish\stockfish-windows-x86-64-universal.exe"
N_WORKERS = max(1, (os.cpu_count() or 2) // 2)


In [3]:
def eval_chunks(fens):
    sf = Stockfish(path=SF_PATH, depth=10, parameters={'Threads': 1, 'Hash': 64})
    return [evaluate_position(sf, fen) for fen in fens]

def evaluate_position(stockfish, fen):
    board = chess.Board(fen)
    if board.is_checkmate():
        return -10000 if board.turn == chess.WHITE else 10000
    if board.is_game_over():
        return 0
    stockfish.set_fen_position(board.fen())
    result = stockfish.get_evaluation()
    if result['type'] == 'cp':
        value = result['value']
    else:
        value = 10000 if result['value'] > 0 else -10000
    return value if board.turn == chess.WHITE else -value

In [4]:
RANDOM_STATE = 42
CV = 5

In [5]:
PIECE_VALUE = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9
}

START_SQUARES = {
    chess.WHITE: {chess.KNIGHT: {chess.B1, chess.G1}, chess.BISHOP: {chess.C1, chess.F1}, chess.QUEEN: {chess.D1}}, chess.BLACK: {chess.KNIGHT: {chess.B8, chess.G8}, chess.BISHOP: {chess.C8, chess.F8}, chess.QUEEN: {chess.D8}}
}

def developed(board, color):
    count = 0
    for piece_type in (chess.KNIGHT, chess.BISHOP):
        for sq in board.pieces(piece_type, color):
            if sq not in START_SQUARES[color][piece_type]:
                count += 1
    return count

def queen_out(board, color): 
    return int(any(sq not in START_SQUARES[color][chess.QUEEN] for sq in board.pieces(chess.QUEEN, color)))

def mobility(board, color):
    b = board.copy(stack=False)
    b.turn = color
    b.ep_square = None
    return b.legal_moves.count()

def material(board, color):
    return sum(PIECE_VALUE.get(p.piece_type, 0)
        for p in board.piece_map().values() if p.color == color)

def extract_features(game, n_moves=10):
    h = game.headers
    base, inc = h['TimeControl'].split('+')
    board = game.board()
    clocks = {chess.WHITE: float('nan'), chess.BLACK: float('nan')}
    plies = 0
    EVAL_PLIES = [10,16,20]
    fens = {}
    last_node = None
    white_castled = 0
    black_castled = 0
    for node in game.mainline():
        mover = board.turn
        if board.is_castling(node.move):
            if mover == chess.WHITE:
                white_castled = 1
            else:
                black_castled = 1
        board.push(node.move)
        c = node.clock()
        if c is not None:
            clocks[mover] = c
        plies += 1
        last_node = node
        if plies in EVAL_PLIES:
            fens[plies] = board.fen()
        if plies == 2 * n_moves:
            break
    if plies < 2 * n_moves:
        return None
    if last_node.is_end():
        return None
    if board.is_game_over():
        return None
    w_elo, b_elo = int(h['WhiteElo']), int(h['BlackElo'])
    return {
        'game_id': h['Site'].split('/')[-1],
        'date': h['UTCDate'], 'time': h['UTCTime'],
        'white': h['White'], 'black': h['Black'], 
        'white_elo': w_elo, 'black_elo': b_elo,
        'elo_diff': w_elo - b_elo,
        'base_sec': int(base), 'inc_sec': int(inc),
        'material_diff': material(board, chess.WHITE) - material(board, chess.BLACK), 'white_clock': clocks[chess.WHITE],
        'black_clock': clocks[chess.BLACK], 'clock_diff': clocks[chess.WHITE] - clocks[chess.BLACK],
        'white_castled': white_castled,
        'black_castled': black_castled,
        'developed_white': developed(board, chess.WHITE),
        'developed_black': developed(board, chess.BLACK),
        'queen_out_white': queen_out(board, chess.WHITE),
        'queen_out_black': queen_out(board, chess.BLACK),
        'mobility_white': mobility(board, chess.WHITE),
        'mobility_black': mobility(board, chess.BLACK),
        'fen_5': fens[10],
        'fen_8': fens[16],
        'fen_10': fens[20],
        'result': h['Result']
    }

def read_games(url, max_games=100, n_moves=10):
    resp = requests.get(url, stream=True)
    rows = []
    stream = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(resp.raw), encoding='utf-8')
    while len(rows) < max_games:
        game = chess.pgn.read_game(stream)
        if game is None:
            break
        try:
            features = extract_features(game, n_moves)
            if features is not None:
                rows.append(features)
        except (KeyError, ValueError):
            continue
    resp.close()
    return pd.DataFrame(rows)

def add_stockfish_eval_parallel(df):
    for fen_col, eval_col in [('fen_5', 'eval_5'), ('fen_8', 'eval_8'), ('fen_10', 'eval_10')]:
        fens = df[fen_col].tolist()
        chunks = [fens[i::N_WORKERS] for i in range(N_WORKERS)]
        with ThreadPoolExecutor(N_WORKERS) as ex:
            results = list(ex.map(eval_chunks, chunks))
        out = [None] * len(fens)
        for i, res in enumerate(results):
            out[i::N_WORKERS] = res
        df[eval_col] = out
    return df

def add_feature(df):
    df['white_clock_rel'] = df['white_clock'] / df['base_sec']
    df['black_clock_rel'] = df['black_clock'] / df['base_sec']
    df['clock_diff_rel'] = df['clock_diff'] / df['base_sec']
    df['elo_mean'] = (df['white_elo'] + df['black_elo']) / 2
    elo_gap = df['elo_diff'].clip(-500, 500)
    df['eval_change_5_8'] = df['eval_8'] - df['eval_5']
    df['eval_change_8_10'] = df['eval_10'] - df['eval_8']
    level = (df['elo_mean'] - df['elo_mean'].mean()) / df['elo_mean'].std()
    df['eval_x_level'] = df['eval_10'] * level
    df['eval_x_gap'] = df['eval_10'] * (elo_gap / 500)
    df['developed_diff'] = df['developed_white'] - df['developed_black']
    df['mobility_diff'] = df['mobility_white'] - df['mobility_black']
    return df

In [ ]:
df = read_games(url='https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst', max_games=50000, n_moves=10)
df.to_csv('games_2017-04_raw.csv', index=False)

df = df[df['result'].isin(['1-0', '0-1'])].copy()
df['white_won'] = (df['result'] == '1-0').astype(int)

df = add_stockfish_eval_parallel(df)
df.to_csv('games_2017-04_eval.csv', index=False)

In [ ]:
df = pd.read_csv('games_2017-04.csv')
for eval in ['eval_5', 'eval_8', 'eval_10']:
    df[eval] = df[eval].clip(-1000, 1000)
    
df = df[df['base_sec'] > 0].copy()
df = add_feature(df)
df = df.replace([np.inf, -np.inf], np.nan)
df.to_csv('games_2017-04_features.csv', index=False)



In [6]:
df = pd.read_csv('games_2017-04_features.csv')
df = df.sort_values(by=['date', 'time']).reset_index(drop=True)

In [7]:
cut = int(len(df) * 0.8)
df_train = df.iloc[:cut].copy()
df_test = df.iloc[cut:].copy()

In [8]:
X_train = df_train.drop(columns=['result', 'white_won'])
y_train = df_train['white_won']
X_test = df_test.drop(columns=['result', 'white_won'])
test_id = df_test['game_id']
y_test = df_test['white_won']

In [9]:
COLUMN_TYPE = {}

for col in df.columns:
    COLUMN_TYPE[col] = 'num'


In [10]:
def split_by_type(columns):
    result = {'num': []}
    for col in columns:
        t = COLUMN_TYPE[col]
        result[t].append(col)
    return result

FEATURE_GROUPS = [
    ('core', ['elo_diff']),
    ('+elo_level', ['white_elo', 'black_elo', 'elo_mean']),
    ('+time_ctrl', ['base_sec', 'inc_sec']),
    ('+clock_raw', ['white_clock', 'black_clock', 'clock_diff']),
    ('+clock_rel', ['white_clock_rel', 'black_clock_rel', 'clock_diff_rel']),
    ('+material', ['material_diff']),
    ('+eval', ['eval_10']),
    ('+eval_trend', ['eval_5', 'eval_8', 'eval_change_5_8', 'eval_change_8_10']),
    ('+eval_x_elo', ['eval_x_level', 'eval_x_gap']),
    ('structure', ['developed_white', 'developed_black', 'developed_diff', 'queen_out_white', 'queen_out_black', 'white_castled', 'black_castled']),
    ('+mobility', ['mobility_white', 'mobility_black', 'mobility_diff'])
]

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [11]:
models =  {
    'xgboost': XGBClassifier(random_state=RANDOM_STATE),
    'logistic': LogisticRegression(max_iter=10000),
    'catboost': CatBoostClassifier(random_state=RANDOM_STATE, verbose=False, allow_writing_files=None), 
    'random_forest': RandomForestClassifier(random_state=RANDOM_STATE)
}

param_distributions = {
    'xgboost': {'model__max_depth': [2,3,4,5], 'model__learning_rate': [0.01, 0.03, 0.1], 'model__n_estimators': [100,300,600], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.6, 0.8, 1.0], 'model__min_child_weight': [1,3,5]},
    'logistic': {'model__C': [0.01, 0.1, 1, 10]},
    'catboost': {'model__max_depth': [3,4,5,6], 'model__learning_rate': [0.03, 0.05, 0.1], 'model__iterations': [300,500,1000], 'model__l2_leaf_reg': [1,3,5,10]},
    'random_forest': {'model__max_depth': [3,4,5,6,8, None], 'model__min_samples_leaf': [1,2,4,8,12], 'model__min_samples_split': [2,5,10,20], 'model__max_features': ['sqrt', 'log2', 0.5], 'model__max_samples': [0.6,0.8,None]}
}

family = {
    'xgboost': 'trees',
    'logistic': 'linear',
    'catboost': 'trees', 
    'random_forest': 'trees'
}


In [12]:
def get_dummy_prediction():
    dummy = DummyClassifier(strategy='constant', constant=1, random_state=RANDOM_STATE)
    score = -cross_val_score(estimator=dummy, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return score

def build_preprocessor(num_cols):
    num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    return ColumnTransformer(transformers=[('num', num_pipeline, num_cols)])

def build_pipeline(preprocessor, model):
    return Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])

def evaluate_model(EXPERIMENT_COLUMNS, model_name, model):
    parts = split_by_type(EXPERIMENT_COLUMNS)
    preprocessor = build_preprocessor(parts['num'])
    pipeline = build_pipeline(preprocessor, model)
    score = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return {'model': model_name, 'score': score.mean()}

def tune_model(COLUMNS, model, model_name, param_distributions, n_iter=30):
    parts = split_by_type(COLUMNS)
    preprocessor = build_preprocessor(parts['num'])
    pipeline = build_pipeline(preprocessor, model)
    search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distributions, n_iter=n_iter, n_jobs=-1, scoring='accuracy', return_train_score=True, refit=False, random_state=RANDOM_STATE, error_score='raise')
    search.fit(X_train, y_train)
    results = pd.DataFrame(search.cv_results_)
    results['model'] = model_name
    results['train_accuracy'] = results['mean_train_score']
    results['validation_accuracy'] = results['mean_test_score']
    best_result = select_best_result(results)
    return search, results, best_result

def select_best_result(results):
    if results.empty:
        raise ValueError('result is empty')
    else:
        return results.loc[results['validation_accuracy'].idxmax()]

def score_columns(chosen, model):
    parts = split_by_type(chosen)
    preprocessor = build_preprocessor(num_cols=parts['num'])
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    score = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    return score.mean()

In [13]:
EXPERIMENT_COLUMNS = []
model_evaluation = []
for model_name, model in models.items():
    model_evaluation.append(evaluate_model(EXPERIMENT_COLUMNS, model_name, model))


KeyboardInterrupt: 

In [32]:
model_evaluation = pd.DataFrame(model_evaluation)
model_evaluation.sort_values(by='score', ascending=False)


,model,score
2,catboost,0.655371
1,logistic,0.652761
3,random_forest,0.647552
0,xgboost,0.612096


In [14]:
rows, used = [], []

for name, cols in FEATURE_GROUPS:
    used = used + cols
    row = {'step': name, 'n_features': len(used)}
    for model_name, model in models.items():
        row[model_name] = score_columns(used, model)
    rows.append(row)
ablation_results = pd.DataFrame(rows)


In [15]:
ablation_results

,step,n_features,xgboost,logistic,catboost,random_forest
0,core,1,0.625066,0.630754,0.629312,0.619325
1,+elo_level,4,0.610674,0.630833,0.626507,0.577094
2,+time_ctrl,6,0.611854,0.630597,0.626219,0.587344
3,+clock_raw,9,0.627949,0.636285,0.639247,0.615812
4,+clock_rel,12,0.625328,0.642367,0.638225,0.619718
5,+material,13,0.634607,0.648972,0.649471,0.629050
6,+eval,14,0.657518,0.667243,0.669078,0.654451
7,+eval_trend,18,0.653245,0.667768,0.666090,0.657833
8,+eval_x_elo,20,0.654241,0.667899,0.666641,0.657780
9,structure,27,0.653272,0.667270,0.666116,0.657649


In [13]:
THRESHOLD = 0.003
model = models['logistic']

final_cols = []
best_score = None
logs = []

for name, cols in FEATURE_GROUPS:
    trial = cols + final_cols
    score = score_columns(trial, model)
    
    if best_score is None or score - best_score > THRESHOLD:
        final_cols = trial
        best_score = score
        logs.append((name, 'KEEP', round(score, 4)))
    else:
        logs.append((name, 'SKIP', round(score,4)))

FINAL_COLUMNS = final_cols 

In [14]:

searches = {}
tuning_results = []
best_results = []
for model_name, model in models.items():
    search, results, best_result = tune_model(FINAL_COLUMNS, model, model_name, param_distributions[model_name], 30)
    searches[model_name] = search
    tuning_results.append(results)
    best_results.append(best_result)
all_tuning_results = pd.concat(tuning_results, ignore_index=True)
best_results = pd.DataFrame(best_results).sort_values(by='validation_accuracy', ascending=False)

c:\Users\yazid\anaconda3\envs\ds_env\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 4 is smaller than n_iter=30. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [ ]:
best_results = pd.read_csv('tuning_results.csv')

In [16]:
best_results.to_csv('tuning_results.csv', index=False)

In [19]:
best_results['family'] = best_results['model'].map(family)
best_models_by_family = best_results.sort_values(by='validation_accuracy', ascending=False).drop_duplicates('family')

In [20]:
best_models_by_family['model']

10    catboost
2     logistic
Name: model, dtype: str

In [21]:
def make_submission(best_models_by_family):
    predictions_proba = []
    parts = split_by_type(FINAL_COLUMNS)
    for _, results in best_models_by_family.iterrows():
        model = clone(models[results['model']])
        preprocessor = build_preprocessor(parts['num'])
        pipeline = build_pipeline(preprocessor, model)
        pipeline.set_params(**results['params'])
        pipeline.fit(X_train, y_train)
        prediction_proba = pipeline.predict_proba(X_test)[:,1]
        predictions_proba.append(prediction_proba)
    avg_proba = np.mean(predictions_proba, axis=0)
    ensemble_prediction = (avg_proba > 0.5).astype(int)
    submission = pd.DataFrame({'game_id': test_id, 'white_won': ensemble_prediction})
    submission.to_csv('./data/submission.csv', index=False)

In [22]:
make_submission(best_models_by_family)

In [ ]:
submission = pd.read_csv('./data/submission.csv')
comparison = df_test[['game_id', 'white_won']].merge(
    submission[['game_id', 'white_won']],
    on='game_id',
    suffixes=('_true', '_pred'),
    validate='one_to_one'
)
y_true = comparison['white_won_true']
y_pred = comparison['white_won_pred']
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))


Accuracy: 0.6770472895040369
Confusion matrix:
 [[3031 1609]
 [1471 3426]]
              precision    recall  f1-score   support

           0       0.67      0.65      0.66      4640
           1       0.68      0.70      0.69      4897

    accuracy                           0.68      9537
   macro avg       0.68      0.68      0.68      9537
weighted avg       0.68      0.68      0.68      9537



In [24]:
pred = np.where(df_test['elo_diff'] > 0, 1, 0)
print('baseline rating:', (pred == y_test).mean())
print('selalu putih menang:', y_test.mean())

baseline rating: 0.63374226696026
selalu putih menang: 0.5134738387333543
